# Rim Center Identification — Ready-to-run Notebook

This notebook contains a lightweight UNet-style heatmap regression pipeline with an **auxiliary visibility classifier** head and combined loss (MSE + BCE). Edit the `images_dir` and `labels_json` paths in the **Usage** cell before running.

In [1]:
# Cell 1 - imports & device selection
import os
from pathlib import Path
import json
import math
import random
from typing import Tuple, List, Optional

import numpy as np
import cv2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

# Select device: prefer MPS on Apple silicon, else CUDA, else CPU
if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print('Using device:', device)


Using device: mps


In [2]:
# Cell 2 - heatmap utility
def make_gaussian_heatmap(h: int, w: int, center: tuple, sigma: float=2.0):
    """Create a normalized gaussian heatmap with peak 1 at `center` (x,y in pixels)."""
    x0, y0 = center
    xs = np.arange(0, w, 1, np.float32)
    ys = np.arange(0, h, 1, np.float32)[:, None]
    d2 = (xs - x0)**2 + (ys - y0)**2
    heat = np.exp(-d2 / (2 * sigma**2)).astype(np.float32)
    heat = heat / (heat.max() + 1e-12)
    return heat


In [3]:
# Cell 3 - Dataset class (handles list or dict JSON formats, normalized centers)
class RimCenterDataset(Dataset):
    def __init__(self, images_dir, labels_json, img_size=(128,128), sigma=2.0, transform=None):
        self.images_dir = Path(images_dir)
        with open(labels_json, 'r') as f:
            raw = json.load(f)
        if isinstance(raw, dict):
            self.entries = [{'filename': k, **v} for k,v in raw.items()]
        elif isinstance(raw, list):
            self.entries = raw
        else:
            raise ValueError('Unsupported labels.json format (expect list or dict)')
        self.img_size = tuple(img_size)
        self.sigma = sigma
        self.transform = transform

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        item = self.entries[idx]
        fname = item.get('filename') or item.get('file_name') or item.get('image')
        img_path = self.images_dir / fname
        img = cv2.imread(str(img_path))
        if img is None:
            raise FileNotFoundError(f'Image not found: {img_path}')
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        orig_h, orig_w = img.shape[:2]

        # resize only if needed
        if (orig_h, orig_w) != self.img_size:
            img_resized = cv2.resize(img, (self.img_size[1], self.img_size[0]), interpolation=cv2.INTER_LINEAR)
        else:
            img_resized = img

        img_resized = img_resized.astype(np.float32) / 255.0
        img_t = torch.tensor(img_resized).permute(2,0,1)  # C,H,W

        is_present = float(item.get('is_present', item.get('visible', 0.0)))
        if is_present:
            cx_norm, cy_norm = item['center']
            px = cx_norm * (self.img_size[1])
            py = cy_norm * (self.img_size[0])
            heat = make_gaussian_heatmap(self.img_size[0], self.img_size[1], (px, py), sigma=self.sigma)
            target = torch.tensor(heat).unsqueeze(0)
            visible = torch.tensor(1.0, dtype=torch.float32)
        else:
            target = torch.zeros((1, self.img_size[0], self.img_size[1]), dtype=torch.float32)
            visible = torch.tensor(0.0, dtype=torch.float32)

        return {'image': img_t, 'heat': target, 'visible': visible, 'meta': {'filename': fname}}


In [4]:
# Cell 4 - Simple UNet with auxiliary visibility head
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )
    def forward(self,x): return self.conv(x)

class Up(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_c, out_c, 2, stride=2)
        self.conv = ConvBlock(in_c, out_c)
    def forward(self,x,skip):
        x = self.up(x)
        # if shapes mismatch due to odd sizes, center-crop skip to x's size (robust)
        if x.shape[2:] != skip.shape[2:]:
            dh = skip.shape[2] - x.shape[2]
            dw = skip.shape[3] - x.shape[3]
            skip = skip[:,:,dh//2:dh//2 + x.shape[2], dw//2:dw//2 + x.shape[3]]
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class SimpleUNetWithVis(nn.Module):
    def __init__(self, in_ch=3, base=32, img_size=(128,128)):
        super().__init__()
        self.enc1 = ConvBlock(in_ch, base)
        self.enc2 = ConvBlock(base, base*2)
        self.enc3 = ConvBlock(base*2, base*4)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(base*4, base*8)

        self.up1 = Up(base*8, base*4)
        self.up2 = Up(base*4, base*2)
        self.up3 = Up(base*2, base)
        self.final = nn.Conv2d(base, 1, kernel_size=1)

        # auxiliary visibility head: global pooled features -> scalar
        self.vis_pool = nn.AdaptiveAvgPool2d((1,1))
        self.vis_fc = nn.Sequential(
            nn.Linear(base*8, base*4),
            nn.ReLU(inplace=True),
            nn.Linear(base*4, 1),
            nn.Sigmoid()
        )
        self.img_size = img_size

    def forward(self,x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))
        u1 = self.up1(b, e3)
        u2 = self.up2(u1, e2)
        u3 = self.up3(u2, e1)
        heat = self.final(u3)
        heat = torch.sigmoid(heat)  # 0..1 heatmap

        # visibility head uses bottleneck features
        v = self.vis_pool(b)  # N, C, 1, 1
        v = v.view(v.shape[0], -1)  # N, C
        vis = self.vis_fc(v).squeeze(1)  # N
        return heat, vis


In [5]:
# Cell 5 - decode & evaluation utilities
import math

def heatmap_to_point(heat: np.ndarray):
    idx = heat.argmax()
    h, w = heat.shape
    y = int(idx // w)
    x = int(idx % w)
    conf = float(heat[y, x])
    return float(x), float(y), conf

def pixel_distance(a: tuple, b: tuple):
    return math.hypot(a[0]-b[0], a[1]-b[1])

def evaluate_model(model, dataloader, device, max_display=4):
    model.eval()
    dists = []
    vis_conf_neg = []
    vis_preds = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Eval'):
            imgs = batch['image'].to(device)
            targets = batch['heat'].cpu().numpy()  # N,1,H,W
            visible = batch['visible'].numpy()
            preds_heat, preds_vis = model(imgs)
            preds_heat = preds_heat.cpu().numpy()
            preds_vis = preds_vis.cpu().numpy()
            N = preds_heat.shape[0]
            for i in range(N):
                pred_h = preds_heat[i,0]
                tar_h = targets[i,0]
                if visible[i] == 1.0:
                    gt_x, gt_y, _ = heatmap_to_point(tar_h)
                    px, py, conf = heatmap_to_point(pred_h)
                    d = pixel_distance((px,py),(gt_x,gt_y))
                    dists.append(d)
                else:
                    vis_conf_neg.append(pred_h.max())
                vis_preds.append(preds_vis[i])
    stats = {}
    if len(dists)>0:
        dists = np.array(dists)
        stats['dist_mean'] = float(dists.mean())
        stats['dist_med'] = float(np.median(dists))
        stats['pct_within_5px'] = float((dists <= 5).mean())
        stats['pct_within_10px'] = float((dists <= 10).mean())
    else:
        stats.update({'dist_mean': None, 'dist_med': None, 'pct_within_5px': None, 'pct_within_10px': None})
    if len(vis_conf_neg)>0:
        stats['neg_mean_conf'] = float(np.mean(vis_conf_neg))
        stats['neg_max_conf'] = float(np.max(vis_conf_neg))
    else:
        stats['neg_mean_conf'] = stats['neg_max_conf'] = None
    # visibility classification AUC-like quick metric: mean of preds on positives vs negatives
    return stats


In [8]:
def train_model_safe(model, train_loader, val_loader, device, epochs=10, lr=1e-3,
                     alpha=1.0, beta=1.0, save_path=None):
    """
    Safe train: ensures save directory exists before torch.save.
    If save_path is None, will save to ./models/rim_unet_vis_best.pth
    """
    if save_path is None:
        save_path = Path.cwd() / "models" / "rim_unet_vis_best.pth"
    else:
        save_path = Path(save_path)

    save_path.parent.mkdir(parents=True, exist_ok=True)  # create parent dirs if needed

    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    mse = nn.MSELoss()
    bce = nn.BCELoss()

    best_val = 1e9
    for epoch in range(1, epochs+1):
        model.train()
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{epochs}')
        for batch in pbar:
            imgs = batch['image'].to(device)
            heat_gt = batch['heat'].to(device)
            vis_gt = batch['visible'].to(device)

            preds_heat, preds_vis = model(imgs)
            loss_heat = mse(preds_heat, heat_gt)
            loss_vis = bce(preds_vis, vis_gt)
            loss = alpha * loss_heat + beta * loss_vis

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += float(loss.item())
            pbar.set_postfix({'loss': running_loss / (pbar.n + 1e-12)})

        # validation
        val_stats = evaluate_model(model, val_loader, device)
        print(f"Epoch {epoch} val: {val_stats}")
        save_metric = val_stats.get('dist_med')
        if save_metric is None:
            # fallback: use running loss average
            save_metric = running_loss / (len(train_loader) if len(train_loader) else 1.0)
        if save_metric < best_val:
            best_val = save_metric
            # Save CPU tensors for portability
            state = {'model_state': {k: v.cpu() for k, v in model.state_dict().items()},
                     'epoch': epoch}
            torch.save(state, str(save_path))
            print('Saved best model to', str(save_path))
    return model

# Example: call the safe training function using a local models/ dir
safe_save = Path.cwd() / "models" / "rim_unet_vis_best.pth"
trained = train_model_safe(model, train_loader, val_loader, device, epochs=6, lr=1e-3,
                           alpha=1.0, beta=1.0, save_path=safe_save)


Epoch 1/6:   0%|          | 0/54 [00:00<?, ?it/s]

Eval:   0%|          | 0/19 [00:00<?, ?it/s]

Epoch 1 val: {'dist_mean': 33.41459263840208, 'dist_med': 28.581841387746717, 'pct_within_5px': 0.02666666666666667, 'pct_within_10px': 0.04666666666666667, 'neg_mean_conf': None, 'neg_max_conf': None}
Saved best model to /Users/amrattia/projects/ATI Perception Take Home Challenge/Challenge 1 - Rim Center Location/models/rim_unet_vis_best.pth


Epoch 2/6:   0%|          | 0/54 [00:00<?, ?it/s]

Eval:   0%|          | 0/19 [00:00<?, ?it/s]

...

Eval:   0%|          | 0/19 [00:00<?, ?it/s]

Epoch 5 val: {'dist_mean': 51.49664268558126, 'dist_med': 48.14545117938028, 'pct_within_5px': 0.013333333333333334, 'pct_within_10px': 0.07333333333333333, 'neg_mean_conf': None, 'neg_max_conf': None}


Epoch 6/6:   0%|          | 0/54 [00:00<?, ?it/s]

Eval:   0%|          | 0/19 [00:00<?, ?it/s]

Epoch 6 val: {'dist_mean': 47.51725876378845, 'dist_med': 44.52455679841677, 'pct_within_5px': 0.02666666666666667, 'pct_within_10px': 0.09333333333333334, 'neg_mean_conf': None, 'neg_max_conf': None}


In [7]:
# Cell 7 - Usage example (edit paths and run)
# Update these paths to point to your local dataset
base_dir = Path.cwd()
images_dir = base_dir / 'labeled_example_data'   # <-- edit if needed
labels_json = images_dir / 'labels.json'         # <-- edit if needed

# params
img_size = (128,128)
sigma = 2.0

dataset = RimCenterDataset(images_dir, labels_json, img_size=img_size, sigma=sigma)
n = len(dataset)
val_frac = 0.15
val_n = int(n * val_frac)
train_n = n - val_n
train_ds, val_ds = random_split(dataset, [train_n, val_n], generator=torch.Generator().manual_seed(42))

# DataLoader safe defaults for macOS / MPS debugging: num_workers=0
use_cuda = torch.cuda.is_available()
pin_memory = True if use_cuda else False
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=0, pin_memory=pin_memory)
val_loader   = DataLoader(val_ds,   batch_size=8,  shuffle=False, num_workers=0, pin_memory=pin_memory)

print('Dataset sizes:', len(train_ds), len(val_ds))
sample = dataset[0]
print('Sample shapes:', sample['image'].shape, sample['heat'].shape, 'visible', sample['visible'])

# Instantiate model
model = SimpleUNetWithVis(in_ch=3, base=32, img_size=img_size)
print('Model parameters (M):', sum(p.numel() for p in model.parameters())/1e6)

# Move to device BEFORE forward
model = model.to(device)

# Quick smoke forward
x = torch.rand(2,3,img_size[0], img_size[1], device=device)
h, v = model(x)
print('Smoke forward shapes -> heat:', h.shape, 'vis:', v.shape)

# Train for a few epochs (tune lr, alpha, beta as needed)
trained = train_model(model, train_loader, val_loader, device, epochs=6, lr=1e-3, alpha=1.0, beta=1.0, save_path='/mnt/data/rim_unet_vis_best.pth')

# After training, evaluate on validation set
stats = evaluate_model(trained, val_loader, device)
print('Final eval stats:', stats)


Dataset sizes: 850 150
Sample shapes: torch.Size([3, 128, 128]) torch.Size([1, 128, 128]) visible tensor(1.)
Model parameters (M): 1.960034
Smoke forward shapes -> heat: torch.Size([2, 1, 128, 128]) vis: torch.Size([2])


Epoch 1/6:   0%|          | 0/54 [00:00<?, ?it/s]

Eval:   0%|          | 0/19 [00:00<?, ?it/s]

Epoch 1 val: {'dist_mean': 33.91680229547097, 'dist_med': 32.14031735997639, 'pct_within_5px': 0.02666666666666667, 'pct_within_10px': 0.06, 'neg_mean_conf': None, 'neg_max_conf': None}


RuntimeError: Parent directory /mnt/data does not exist.

In [ ]:
# Cell 8 - helper to visualize predictions for a few validation samples
def show_predictions(model, dataset, device, n=6):
    model.eval()
    fig = plt.figure(figsize=(12, 2*n))
    shown = 0
    with torch.no_grad():
        for i in range(len(dataset)):
            sample = dataset[i]
            img = sample['image'].unsqueeze(0).to(device)
            heat_gt = sample['heat'].numpy()[0]
            vis_gt = sample['visible'].item()
            pred_h, pred_v = model(img)
            pred_h = pred_h.cpu().numpy()[0,0]
            pred_v = float(pred_v.cpu().numpy()[0])
            ax = plt.subplot(n, 3, shown*3 + 1)
            ax.imshow(sample['image'].permute(1,2,0).numpy())
            ax.set_title(f'input - vis_gt={vis_gt}')
            ax.axis('off')
            ax = plt.subplot(n, 3, shown*3 + 2)
            ax.imshow(heat_gt, vmin=0, vmax=1)
            ax.set_title('gt heatmap')
            ax.axis('off')
            ax = plt.subplot(n, 3, shown*3 + 3)
            ax.imshow(pred_h, vmin=0, vmax=1)
            ax.set_title(f'pred heat - vis_pred={pred_v:.2f}')
            ax.axis('off')
            shown += 1
            if shown >= n:
                break
    plt.tight_layout()
    plt.show()

# Example usage (run after training)
# show_predictions(trained, val_ds, device, n=4)
